# AdaBoost-FKD example

In this notebook, we present how to execute the AdaBoost-FKD algoritm, as well as the local version without the federated process, for comparison purposes.
It is assumed that with this example, the rest of experiments could be also addressed.

In [1]:
# Import data from flextrees
# TO-DO: Habria que poner algun ejemplo que no use dataset de flex?
from flextrees.datasets.tabular_datasets import adult
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# Import AdaBoostFKD, and also the Federated Random Forest, for comparison
from models.AdaBoostFKD import AdaBoostFKD
from models.FRF import FRF_eval

from sklearn.datasets import load_breast_cancer

In [2]:
# Fix the random numbers seed
seed = 0

Load the dataset.

Note that the whole dataset (both train and test) is loaded and joined together. Later, the whole data is distributed among the clients to simulate the federated scenario.

In [3]:
train_data, test_data = adult(ret_feature_names=False, categorical=False)
X_data,y_data = train_data.to_numpy()
X_test,y_test = test_data.to_numpy()
X_data = np.concatenate((X_data,X_test))
y_data = np.concatenate((y_data,y_test))

#Alternatively, you can directly use any numpy dataset of your own:

#bcancer = load_breast_cancer()
#X_data = bcancer.data
#y_data = bcancer.target

In [4]:
# Separate public data (a small portion, for example, 5%)
# It is considered to be unlabeled, so we do not store the targets of the public data
data, public_data, targets, _ = train_test_split(X_data, y_data, test_size=0.05, random_state=seed)

# Get global test set (i.e., 10%), and the rest of training data that will be later distributed among clients
X_train, X_test, y_train, y_test = train_test_split(data,targets,test_size=0.1,random_state=seed)

Run AdaBoost-FKD

In [5]:
# When creating the model the data partition according to the chosen data distribution is made. 
fl_model = AdaBoostFKD(X_train, y_train, public_data, 
                         n_clients=10, T=10,
                         data_distribution='niid_quantity_skew', distribution_param=0.5,
                         public_data_prediction='weighted_majority_voting', 
                         server_alpha_weight_adj='common_weighted',
                         prediction_weights='only_server',soft_predictions=True, temperature=3,
                         client_weight_adj='common',    
                         random_state=seed,
                         server_classifier=DecisionTreeClassifier, server_classifier_params={'random_state':seed, 'max_depth':4, 'max_leaf_nodes':None},
                         clients_classifier=DecisionTreeClassifier, clients_classifier_params={'random_state':seed, 'max_depth':4, 'max_leaf_nodes':None})

# Store data distrib for subsequent models
train_dict = fl_model.train_clients_data.copy()
test_dict = fl_model.test_clients_data.copy()

# Takes the columns of weights of AdaBoost out
for key,(train,labeltr) in train_dict.items():
    train_dict[key] = (train[:,:-1],labeltr)
    test,labelte = test_dict[key]
    test_dict[key] = (test[:,:-1],labelte)

In [6]:
#Train the model
fl_model.fitmodel()

In [7]:
# Evaluate how the federated model works in local tests and global test (in average among clients) 
fl_acc_global, fl_f1_global, fl_auc_global, fl_acc_local, fl_f1_local, fl_auc_local = fl_model.overall_score(X_test, y_test)

print(f'AdaBoost-FKD accuracy (global test): {fl_acc_global:.4f}')
print(f'AdaBoost-FKD f1-score (global test): {fl_f1_global:.4f}')
print(f'AdaBoost-FKD accuracy (local test): {fl_acc_local:.4f}')
print(f'AdaBoost-FKD f1-score (local test): {fl_f1_local:.4f}')
print(f'AdaBoost-FKD AUC (global test): {fl_auc_global:.4f}')
print(f'AdaBoost-FKD AUC (local test): {fl_auc_local:.4f}')

AdaBoost-FKD accuracy (global test): 0.8568
AdaBoost-FKD f1-score (global test): 0.8619
AdaBoost-FKD accuracy (local test): 0.8338
AdaBoost-FKD f1-score (local test): 0.8526
AdaBoost-FKD AUC (global test): 0.9011
AdaBoost-FKD AUC (local test): 0.9032


Run local AdaBoost models at each client

In [8]:
# If we want to compare it to the local models (without federated process), we first need to train local models with same data
fl_model.fit_local_clients_models()

In [9]:
# Then we can get a Dataframe with all the results for each client
localAB_acc_scores = fl_model.overall_acc_score(X_test, y_test)
localAB_acc_scores

,data_distrib,FL_acc_own_data,FL_acc_global_data,local_acc_own_data,local_acc_global_data,local_difference,global_difference
0,1596.0,0.840000,0.85682,0.857500,0.842599,-0.017500,0.014221
1,534.0,0.888060,0.85682,0.820896,0.822883,0.067164,0.033937
2,1051.0,0.851711,0.85682,0.840304,0.826115,0.011407,0.030705
3,184.0,0.869565,0.85682,0.826087,0.820944,0.043478,0.035876
4,9.0,0.666667,0.85682,0.333333,0.593407,0.333333,0.263413
5,5717.0,0.841259,0.85682,0.839161,0.845831,0.002098,0.010989
6,4991.0,0.834135,0.85682,0.832532,0.845507,0.001603,0.011312
7,1163.0,0.838488,0.85682,0.828179,0.840659,0.010309,0.016160
8,2674.0,0.862481,0.85682,0.844544,0.840336,0.017937,0.016484
9,4344.0,0.845304,0.85682,0.843462,0.842922,0.001842,0.013898


In [10]:
localAB_f1_scores = fl_model.overall_F1_score(X_test,y_test)
localAB_f1_scores

,data_distrib,FL_wf1_own_data,FL_wf1_global_data,local_wf1_own_data,local_wf1_global_data,local_difference_w,global_difference_w
0,1596.0,0.843684,0.861899,0.863437,0.850937,-0.019752,0.010962
1,534.0,0.890324,0.861899,0.825855,0.831018,0.064469,0.030881
2,1051.0,0.858419,0.861899,0.875813,0.854707,-0.017395,0.007192
3,184.0,0.875049,0.861899,0.843249,0.832395,0.031800,0.029504
4,9.0,0.800000,0.861899,0.333333,0.572620,0.466667,0.289279
5,5717.0,0.848074,0.861899,0.852045,0.857497,-0.003971,0.004402
6,4991.0,0.842182,0.861899,0.846077,0.855388,-0.003895,0.006511
7,1163.0,0.845697,0.861899,0.835252,0.846508,0.010445,0.015391
8,2674.0,0.869712,0.861899,0.854868,0.848849,0.014844,0.013050
9,4344.0,0.852780,0.861899,0.856156,0.852399,-0.003377,0.009500


In [11]:
# Evaluate how the local models works in local tests and global test (in average among them) 
localAB_acc_global = localAB_acc_scores['local_acc_global_data'].mean()
localAB_f1_global = localAB_f1_scores['local_wf1_global_data'].mean()
localAB_acc_local = localAB_acc_scores['local_acc_own_data'].mean()
localAB_f1_local = localAB_f1_scores['local_wf1_own_data'].mean()

print(f'AdaBoost-FKD accuracy (global test): {localAB_acc_global:.4f}')
print(f'AdaBoost-FKD f1-score (global test): {localAB_f1_global:.4f}')
print(f'AdaBoost-FKD accuracy (local test): {localAB_acc_local:.4f}')
print(f'AdaBoost-FKD f1-score (local test): {localAB_f1_local:.4f}')

AdaBoost-FKD accuracy (global test): 0.8121
AdaBoost-FKD f1-score (global test): 0.8202
AdaBoost-FKD accuracy (local test): 0.7866
AdaBoost-FKD f1-score (local test): 0.7986


Compare to state-of-the-art methods

In [12]:
# Now we can use it to compare it to other state of the art algorithms such as FRF with 100 estimators
FRF_acc_global, FRF_f1_global, FRF_acc_local, FRF_f1_local = FRF_eval(train_dict, test_dict, X_test, y_test, hyperparameters='theirs')

In [13]:
print(f'FRF model accuracy (global test): {FRF_acc_global:.4f}')
print(f'FRF model f1-score (global test): {FRF_f1_global:.4f}')
print(f'FRF model accuracy (local test): {FRF_acc_local:.4f}')
print(f'FRF model f1-score (local test): {FRF_f1_local:.4f}')

FRF model accuracy (global test): 0.8342
FRF model f1-score (global test): 0.8147
FRF model accuracy (local test): 0.8257
FRF model f1-score (local test): 0.7938
